In [37]:
import pandas as pd
import numpy as np
from importlib import reload
import help

reload(help)

train = help.fetch_data('train.csv')
test = help.fetch_data('test.csv')
holidays_events = help.fetch_data('holidays_events.csv')
oil = help.fetch_data('oil.csv')
stores = help.fetch_data('stores.csv')
transactions = help.fetch_data('transactions.csv')




In [38]:
train.describe()

,id,store_nbr,sales,onpromotion
count,3.000888e+06,3.000888e+06,3.000888e+06,3.000888e+06
mean,1.500444e+06,2.750000e+01,3.577757e+02,2.602770e+00
std,8.662819e+05,1.558579e+01,1.101998e+03,1.221888e+01
min,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00
25%,7.502218e+05,1.400000e+01,0.000000e+00,0.000000e+00
50%,1.500444e+06,2.750000e+01,1.100000e+01,0.000000e+00
75%,2.250665e+06,4.100000e+01,1.958473e+02,0.000000e+00
max,3.000887e+06,5.400000e+01,1.247170e+05,7.410000e+02


In [39]:
train.set_index('id', inplace=True)
test.set_index('id', inplace=True)
print(train.info(), holidays_events.info(), stores.info(), oil.info(), transactions.info())

<class 'pandas.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 5 columns):
 #   Column       Dtype  
---  ------       -----  
 0   date         str    
 1   store_nbr    int64  
 2   family       str    
 3   sales        float64
 4   onpromotion  int64  
dtypes: float64(1), int64(2), str(2)
memory usage: 114.5 MB
<class 'pandas.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   date         350 non-null    str  
 1   type         350 non-null    str  
 2   locale       350 non-null    str  
 3   locale_name  350 non-null    str  
 4   description  350 non-null    str  
 5   transferred  350 non-null    bool 
dtypes: bool(1), str(5)
memory usage: 14.1 KB
<class 'pandas.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   store_nbr  54 non-null    

In [40]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('encoder', OneHotEncoder())
])

In [41]:
train['date'] = pd.to_datetime(train['date'])
test['date'] = pd.to_datetime(test['date'])
for df in [train, test]:
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['dayofweek'] = df['date'].dt.day_of_week
    df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)
    df['isweekend'] = df['dayofweek'].isin([5, 6]).astype(int)

In [42]:
train = train.merge(stores, on='store_nbr', how='left')
test = test.merge(stores, on='store_nbr', how='left')
X = train.drop(columns=['sales'])

In [43]:
from sklearn.compose import ColumnTransformer

num_attribs = ['store_nbr', 'onpromotion', 'year', 'month', 'day', 'dayofweek', 'weekofyear', 'isweekend']
cat_attribs = ['family', 'city', 'state', 'type', 'cluster']

full_pipeline = ColumnTransformer([
    ('num', num_pipeline, num_attribs),
    ('cat', cat_pipeline, cat_attribs)
])

X_train = full_pipeline.fit_transform(X)

In [44]:
y = train['sales']
X_test = full_pipeline.transform(test)


In [45]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()

lin_reg.fit(X_train, y)
lin_pred = lin_reg.predict(X_test)

In [46]:
help.create_submission_csv(lin_pred, test.index)